<a href="https://colab.research.google.com/github/kennethkvs/RetinalDiseasesOCTClassifier/blob/main/RetinalDiseaseOCTClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retinal Disease OCT Classifier (COMPSCI 4AL3 Assignment)


In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
from PIL import Image
import sklearn
import matplotlib.pyplot as plt

# PyTorch Imports
import torch
from torchvision.transforms import v2 as image_preprocess
from torch.utils.data import DataLoader
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim

In [ ]:
# Dataset Class Definition
class OCTImageDataset:
    def __init__(self, data_dir: str, type: str):
        self.data_dir = data_dir
        self.type = type
        self.label_mapping = {
            'CNV': 0,
            'DME': 1,
            'DRUSEN': 2,
            'NORMAL': 3
        }
        self.image_paths = []
        self.classes = list(self.label_mapping.keys())
        self.labels = []
        self.image_processor = None

        # Initialize class attributes
        match type:
            case 'train':
                self.image_processor = image_preprocess.Compose([
                    # Convert to tensor since input is a PIL image
                    image_preprocess.ToImage(), 
                    image_preprocess.ToDtype(torch.uint8, scale=True),
                    # Crop
                    image_preprocess.Resize((224, 224), antialias=True),
                    # Different transformations
                    image_preprocess.RandomHorizontalFlip(),
                    # Normalize
                    image_preprocess.ToDtype(torch.float32, scale=True),  # Normalize expects float input according to docs
                    image_preprocess.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
                ])
            case 'test' | 'val':
                self.image_processor = image_preprocess.Compose([
                    # Convert to tensor since input is a PIL image
                    image_preprocess.ToImage(), 
                    image_preprocess.ToDtype(torch.uint8, scale=True),
                    # Crop
                    image_preprocess.Resize((224, 224), antialias=True),
                    # Normalize
                    image_preprocess.ToDtype(torch.float32, scale=True),  # Normalize expects float input according to docs
                    image_preprocess.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
                ])
            case _:
                raise ValueError("Type must be 'train', 'val', or 'test'.")
        self.load_data()

    def load_data(self):
        for label in os.listdir(self.data_dir):
            label_dir = os.path.join(self.data_dir, label)
            if os.path.isdir(label_dir):
                for image_file in os.listdir(label_dir):
                    if image_file.endswith(('.png', '.jpg', '.jpeg')):
                        image_path = os.path.join(label_dir, image_file)
                        self.image_paths.append(image_path)
                        self.labels.append(self.label_mapping[label])
        
        print(f"Loaded {len(self.image_paths)} images for {self.type} set.")

    def visualize(self):
        # Print out dataset statistics
        class_counts = {
            'CNV': self.labels.count(0),
            'DME': self.labels.count(1),
            'DRUSEN': self.labels.count(2),
            'NORMAL': self.labels.count(3)
        }
        print(f"Visualizing dataset statistics for {self.type} set:")
        for class_name, count in class_counts.items():
            print(f"{class_name}: {count} images")
        
        # Plot class distribution
        plt.bar(class_counts.keys(), class_counts.values())
        plt.title(f'Class Distribution in {self.type} Set')
        plt.xlabel('Classes')
        plt.ylabel('Number of Images')
        plt.show()

        # Display sample images from each class
        plt.figure(figsize=(12, 6))
        for class_name in self.classes:
            # Pick the first image from each class for visualization
            image_path = os.path.join(self.data_dir, class_name, os.listdir(os.path.join(self.data_dir, class_name))[0])
            image = Image.open(image_path)
            plt.subplot(1, 4, self.label_mapping[class_name] + 1)
            plt.imshow(image, cmap='gray') # Image data is grayscale idk why without cmap it isn't grayscale
            plt.title(class_name)
        plt.show()

    # Needed for PyTorch DataLoader compatibility
    def __len__(self):
        if len(self.image_paths) != len(self.labels):
            raise ValueError("Mismatch between number of images and labels.")
        if len(self.image_paths) == 0 or len(self.labels) == 0:
            raise ValueError("No images found in the dataset.")

        return len(self.image_paths)
    
    # Needed for PyTorch DataLoader compatibility
    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = self.labels[idx]

        image = Image.open(image_path).convert('RGB')
        if self.image_processor:
            image = self.image_processor(image)
        else:
            raise ValueError("Image processor not defined.")
        
        return image, label


In [ ]:
# Data Configurations
batch_size = 64
shuffle_data = True
# train_data_dir = 'Dataset/train'
# test_data_dir = 'Dataset/test'
# val_data_dir = 'Dataset/val'

train_data_dir = '1000_images_only/train'
test_data_dir = '1000_images_only/test'
val_data_dir = '1000_images_only/val'

In [ ]:
# Training Dataset Initialization
training_data = OCTImageDataset(data_dir=train_data_dir, type='train')
training_data.visualize()
print(f'Sample training data tensor\n{training_data[0]}')

# Training Data Loader
training_data_loader = DataLoader(dataset=training_data, batch_size=batch_size, shuffle=shuffle_data)

In [ ]:
# Testing Dataset Initialization
testing_data = OCTImageDataset(data_dir=test_data_dir, type='test')
testing_data.visualize()
print(f'Sample testing data tensor\n{testing_data[0]}')

# Testing Data Loader
testing_data_loader = DataLoader(dataset=testing_data, batch_size=batch_size, shuffle=shuffle_data)

In [ ]:
# Validation Dataset Initialization
val_data = OCTImageDataset(data_dir=val_data_dir, type='val')
val_data.visualize()
print(f'Sample validation data tensor\n{val_data[0]}')

# Training Data Loader
val_data_loader = DataLoader(dataset=val_data, batch_size=batch_size, shuffle=shuffle_data)

### Machine Learning Model: ResNet Transfer Learning

Following the guide: https://www.geeksforgeeks.org/deep-learning/how-to-implement-transfer-learning-in-pytorch/

- Using a ResNet-50 implementation with a pre-trained ResNet-50 model.

In [ ]:
# Selecting the graphics card is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load the pre-trained ResNet-50 model
# model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1) # Using the new weights parameter as 'pretrained=True' is deprecated

# Model Initialization: Pre-trained ResNet-50 with Modified Final Layer
# class ModifiedResNet(nn.Module):
#     def __init__(self):
#         super(ModifiedResNet, self).__init__()
#         self.resnet = torch.hub.load('pytorch/vision', 'resnet50', weights=models.ResNet50_Weights.IMAGENET1K_V1)
#         # Modify the final layer to match the number of classes (4 in this case)
#         self.resnet.fc = nn.Linear(self.resnet.fc.in_features, 4)

#     def forward(self, x):
#         return self.resnet(x)

# Instantiate the ModifiedResNet model
# pretrained_model = ModifiedResNet()

# Defining the Model Architecture with Transfer Learning
class CustomResNet(nn.Module):
    def __init__(self,):
        super(CustomResNet, self).__init__()
        self.resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(self.resnet.children())[:-1])  # All layers except the final FC layer
        self.classifier = nn.Linear(self.resnet.fc.in_features, 4)  # New final layer for 4 classes

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# Instantiate the CustomResNet model, passing the pretrained model
model = CustomResNet().to(device)

# Freeze all layers except the final layer
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last layer of the ResNet
for param in model.resnet.layer4.parameters():
    param.requires_grad = True

# Unfreeze the classifier layer allowing for fine-tuning
for param in model.classifier.parameters():
    param.requires_grad = True

# Compiling the Model
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)  # Only optimize the final layer parameters

print(f'Model ready for training:\n{model}')

In [ ]:
## Training the Model ##
# Only trains if model doesn't already exist

if os.path.exists('Trained_Model/trained_resnet50_1000_images.pth'):
    # Extracting the model and naming it model so it can be fine tuned and evaluated
    model_path = 'Trained_Model/trained_resnet50_1000_images.pth'
    model.load_state_dict(torch.load(model_path, map_location=device))

    # Moving model to the appropriate device
    model.to(device)

    print("Trained model already exists. Skipping training.")
else:
    # Initialization parameters
    num_epochs = 10
    train_losses = []

    # Training accuracy counters
    train_correct = 0
    train_total = 0

    # Training Loop
    for epoch in range(num_epochs):
        # Set model to training mode
        # Dropout is turned on and batchnorm updates its running statistics
        model.train()
        
        # Reset running loss, correct training amount, and total train ammount for each epoch
        running_loss = 0.0

        # Iterate over training data
        for images, labels in training_data_loader:
            # Move data to the appropriate device
            images, labels = images.to(device), labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Running forward pass
            outputs = model(images)

            # Compute loss and backpropagate
            loss = criterion(outputs, labels)
            loss.backward()

            # Update model parameters
            optimizer.step()

            # Update running loss and accuracy
            running_loss += loss.item()

            # Extracts predicted class index for each sample in the batch
            _, predicted = torch.max(outputs.data, 1)

            # Adds batch size to the running total number of samples processed
            train_total += labels.size(0)

            # Adds number of correct predictions to running total
            train_correct += (predicted == labels).sum().item()

        # Compute average loss and accuracy for the epoch
        train_losses.append(running_loss / len(training_data_loader))
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss / len(training_data_loader):.4f}')

    train_accuracy = train_correct / train_total
    print(f'\n-------------------------------------------------------------------------------------')
    print(f'Finished fine-tuning with {train_accuracy:.4f} training accuracy.')
    print(f'-------------------------------------------------------------------------------------')

    # Saving the model
    model_save_path = 'Trained_Model/trained_resnet50_1000_images.pth'
    torch.save(model.state_dict(), model_save_path)
    print(f'\nModel saved to {model_save_path}')

In [ ]:
## Fine Evaluation on Validation Set ##

# Initialization parameters
num_epochs = 10
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Validation correct and total counters
val_correct = 0
val_total = 0

# Validation Loop
for epoch in range(num_epochs):
    # Set model to evaluation mode (Validation)
    # Dropout is turned off and batchnorm uses running statistics instead of updating them
    model.eval()

    # Reset running loss for each epoch
    val_loss = 0.0

    # Iterate over validation data while stopping gradient computation
    # This is important to prevent updating model parameters during validation
    # It also reduces memory consumption
    with torch.no_grad():
        for images, labels in val_data_loader:
            # Move data to the appropriate device
            images, labels = images.to(device), labels.to(device)

            # Running forward pass
            outputs = model(images)

            # Compute loss
            loss = criterion(outputs, labels)
            
            # Adds scalar loss value to running total, with .item converting the tensor to a Python float
            val_loss += loss.item()

            # Extracts predicted class index for each sample in the batch
            _, predicted = torch.max(outputs.data, 1)

            # Adding batch size to the running total number of samples processed
            val_total += labels.size(0)

            # Compares predicted labels to true labels and sums correct predictions
            val_correct += (predicted == labels).sum().item()

    # Compute average lossfor the epoch
    val_loss_accuracy = val_loss / len(val_data_loader)
    val_losses.append(val_loss_accuracy)

    print(f'Epoch [{epoch+1}/{num_epochs}], Validation Loss: {val_loss_accuracy:.4f}')

# Compute average validation accuracy over all epochs
val_accuracy = val_correct / val_total
print(f'\n-------------------------------------------------------------------------------------')
print(f'Finished validation with {val_accuracy:.4f} validation accuracy.')
print(f'-------------------------------------------------------------------------------------')

In [ ]:
## Evaluating the Model ##

# Initialize variables for tracking performance
test_losses = []
epoch = 10

# Correct and total for accuracy calculation
correct = 0
total = 0

# Testing Loop
for epoch in range(num_epochs):
    # Set model to evaluation mode (Testing)
    model.eval()

    # Reset running loss for each epoch
    running_loss = 0.0

    # Iterate over testing data while stopping gradient computation
    # This is important to prevent updating model parameters during testing
    # It also reduces memory consumption
    with torch.no_grad():

        # Evaluate on test data
        for images, labels in testing_data_loader:
            # Move data to the appropriate device
            images, labels = images.to(device), labels.to(device)

            # Get model outputs
            outputs = model(images)

            # Compute loss (assuming your loss function is defined)
            loss = criterion(outputs, labels)

            # Update running loss
            running_loss += loss.item()

            # Calculate accuracy
            _, predicted = torch.max(outputs.data, 1) # Get the index of the maximum value
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    # Calculate average loss for the epoch
    test_loss = running_loss / len(testing_data_loader)
    test_losses.append(test_loss)

    # Print epoch-wise performance
    print(f'Epoch [{epoch+1}/{num_epochs}], Test Loss: {test_loss:.4f}, Test Accuracy: {correct / total:.4f}')

# Final Test Accuracy
test_accuracy = correct / total
print(f'\n-------------------------------------------------------------------------------------')
print(f'Final Test Accuracy: {test_accuracy:.4f} testing accuracy.')
print(f'-------------------------------------------------------------------------------------')